# Greenplum

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/greenplum/greenplum.ipynb)

Greenplum Database is an open-source, massively parallel (MPP) analytical database built on PostgreSQL. A coordinator plans queries, segments store and process data in parallel, and an interconnect moves rows between them. These interview questions cover architecture, distribution and skew, EXPLAIN ANALYZE, gpfdist and external tables, partitioning, AO/CO storage, VACUUM, and resource groups.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"duckdb": "duckdb>=1.0"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

Greenplum needs a cluster, so the runnable cells use DuckDB as an in-process stand-in and compute the segment each row would hash to. That is enough to see skew, motion and columnar compression with real numbers.

## Architecture

The mental model is one PostgreSQL coordinator in front of many PostgreSQL segment instances that all run the same query at the same time. Greenplum 6 is built on PostgreSQL 9.4 and Greenplum 7 on PostgreSQL 12, so almost all PostgreSQL SQL, types, and client drivers work unchanged. What is new is that every table is physically split across segments and the planner must decide how to move rows between them.

Each segment host runs several segment instances, each with its own data directory and port. The coordinator stores no user data; it parses, plans, dispatches plan slices, and merges results. The number of segment instances is the parallelism budget, and every operator runs once per segment.

Common mistake: treating Greenplum as a bigger PostgreSQL and loading one huge file through the coordinator. That funnels data through a single node and leaves the segments idle.

```text
client -> coordinator (parse, plan, dispatch slices)
              |
        interconnect (UDP with IC proxy, or TCP)
        /            |            \
   segment 0     segment 1     segment N     (each with a mirror on another host)
```

The coordinator is the single entry point. It holds `pg_catalog`, accepts connections, plans every statement, and runs the final Gather and merge. It does not scan user tables. Segments are independent PostgreSQL instances that store disjoint slices of every distributed table; there are usually several per host, one per CPU socket or vCPU group. The interconnect is the network fabric, now UDP with flow control for data and TCP for control messages, over which motion operators move rows.

Queries are cut into slices. A slice is a fragment of the plan that runs on every segment in parallel, so a plan with four slices does four passes of parallel work, often pipelined. Mirrors are standby segment instances on a different host that replay the primary's write-ahead log and are promoted if a primary fails. `gp_segment_configuration` is the source of truth for which segment is primary, mirror, or down.

Common mistake: expecting the coordinator to be the bottleneck. It mostly is not, except for plans that Gather huge result sets or that broadcast large tables.

### Parallel execution and slices

The planner produces a directed acyclic plan and then cuts it into slices at motion boundaries. A slice runs on all segments at once, so parallelism equals the number of segment instances, not the number of rows. Some slices are pipelined, one producer and one consumer running together; others are materialized when the plan needs a barrier.

```sql
EXPLAIN ANALYZE
SELECT p.category, count(*) AS sales
FROM fact_sales s
JOIN dim_product p ON p.product_id = s.product_id
WHERE s.sale_date >= date '2026-01-01'
GROUP BY p.category;
```

Read the plan bottom-up. A `Slice` number appears next to each node. `rows` is the estimate; under `EXPLAIN ANALYZE` the actual rows per slice are shown, and the executor reports the max across segments. When one slice's max is far above the average, work is not balanced and the whole query waits for the slowest segment.

Common mistake: reading only total runtime and ignoring the per-slice distribution, which is where the real bottleneck hides.

### Interconnect and dispatcher

The interconnect is the network layer that carries motion data. When a plan needs rows moved, the sending slice opens connections to the receiving slice on each target segment and streams tuples. Data flows over UDP with flow control for throughput, while control messages use TCP. The coordinator never sees intermediate motion traffic; it only receives the final results.

The performance consequence is that interconnect bandwidth is a shared resource. Two concurrent queries each redistributing a billion rows compete for the same fabric, and the slower one delays the faster one. Broadcast of a large table is the classic way to saturate it.

```sql
-- Look for the receiving and sending slice ids and the row counts
EXPLAIN ANALYZE
SELECT c.region, sum(s.revenue)
FROM fact_sales s JOIN dim_customer c ON c.customer_id = s.customer_id
GROUP BY c.region;
```

Common mistake: reading only local disk I/O when a plan is slow. On MPP, the bottleneck is often rows crossing the interconnect, and that does not appear in `BUFFERS`.

### Greenplum versus PostgreSQL

| Aspect | PostgreSQL | Greenplum |
|---|---|---|
| Execution | single node | coordinator + segments (MPP) |
| Storage | heap row | heap plus AO row/column |
| Joins | local | local or Motion (redistribute/broadcast) |
| Best at | OLTP | large analytical scans |
| Extras | broad extensions | MADlib, PXF, gpfdist, resource groups |

## Distribution

### Distribution policies

Distribution is chosen at CREATE TABLE time and stored in `pg_class.policy` and `gp_distribution_policy`. A hash policy applies a hash function to the distribution key and maps the result to a segment id, so equal keys always land on the same segment. That is what lets a join on the distribution key run segment-local with no motion.

```sql
CREATE TABLE fact_orders (
  order_id    bigint,
  customer_id bigint,
  order_ts    timestamp,
  amount      numeric(12,2)
)
DISTRIBUTED BY (customer_id);
```

A random policy spreads rows round-robin at insert time, giving even sizes but no join locality. A replicated policy copies the whole table to every segment, which removes motion for small tables but multiplies storage and write cost by the segment count. Distribution is fixed at create time; change it with `ALTER TABLE ... SET DISTRIBUTED BY (...)`, which rewrites and reshuffles the whole table.

Common mistake: picking a low-cardinality distribution key such as `status` or `country`. A handful of hash values collapse onto a few segments.

### Choosing a distribution key

A distribution key has to satisfy three properties at once: high cardinality, even value frequency, and relevance to the dominant join. The first two make rows spread evenly across segments, so no segment is a straggler. The third makes the busiest join segment-local, so no rows cross the interconnect. If a column is good at one and bad at the others, it is usually the wrong key.

Start from the workload, not the schema. List the joins in the top twenty queries by runtime, weight them by frequency, and pick the column that appears most often on both sides of those joins. Then check its cardinality with `count(distinct ...)` and its frequency distribution with a top-values query. Only after that does the distribution key earn its place.

```sql
-- Join relevance
SELECT c.customer_id, count(*) AS rows
FROM fact_orders o JOIN dim_customer c ON c.customer_id = o.customer_id
GROUP BY 1 ORDER BY 2 DESC LIMIT 5;

-- Cardinality and frequency
SELECT count(DISTINCT customer_id) AS distinct_keys,
       count(*) AS total_rows,
       count(*)::numeric / count(DISTINCT customer_id) AS rows_per_key
FROM fact_orders;

SELECT customer_id, count(*) AS rows
FROM fact_orders GROUP BY 1 ORDER BY 2 DESC LIMIT 10;
```

Common mistake: choosing the primary key of the fact table. A surrogate `order_id` has perfect cardinality and is useless for joins, so every dimension join redistributes

When two large tables join often, distributing both on the join key removes the motion. When a small dimension joins a large fact, replicate the dimension instead: it is copied to every segment, so no motion is needed and the storage cost is small. When no column is dominant, distribute randomly and accept the redistribution.

```sql
CREATE TABLE dim_customer (
  customer_id bigint PRIMARY KEY,
  country     text,
  segment     text
) DISTRIBUTED BY (customer_id);

CREATE TABLE fact_orders (
  order_id    bigint,
  customer_id bigint,
  order_ts    timestamp,
  amount      numeric(12,2)
) DISTRIBUTED BY (customer_id);

-- Small lookup that joins every fact: replicate it instead
CREATE TABLE dim_currency (code text, name text) DISTRIBUTED REPLICATED;
```

Confirm the effect with `EXPLAIN`: a join on the shared key should show no `Redistribute Motion` or `Broadcast Motion` above it. If motion remains, check that both tables really use the same key and that the planner is not estimating one side badly.

The cells below build an order table and hash it onto eight virtual segments with three candidate keys.

In [ ]:
import tempfile
from pathlib import Path

import duckdb

SEGMENTS = 8
con = duckdb.connect()
con.execute("""
    CREATE TABLE fact_orders AS
    SELECT i AS order_id,
           (i * 7919) % 5000 AS customer_id,
           ['US', 'US', 'US', 'US', 'DE', 'IN', 'BR'][1 + i % 7] AS country,
           ['shipped', 'pending'][1 + (i % 10 = 0)::INT] AS status,
           round((i % 400) * 1.25, 2) AS amount
    FROM range(200000) t(i)
""")
con.execute("CREATE TABLE dim_customer AS SELECT i AS customer_id, 'tier_' || (i % 4) AS tier FROM range(5000) t(i)")


def segment_of(column):
    """DISTRIBUTED BY (column): hash the key and map it to a segment id."""
    return f"hash({column}) % {SEGMENTS}"

In [ ]:
def skew_report(column):
    """Rows per segment and a skew coefficient (stddev / mean), like gp_toolkit.gp_skew_coefficients."""
    expression = "(random() * 1e9)::BIGINT % 8" if column == "RANDOMLY" else segment_of(column)
    return con.execute(f"""
        WITH per_segment AS (
            SELECT {expression} AS gp_segment_id, count(*) AS n FROM fact_orders GROUP BY 1
        )
        SELECT count(*) AS used_segments, max(n) AS max_rows, round(stddev_pop(n) / avg(n), 3) AS skew
        FROM per_segment
    """).fetchone()


report = {key: skew_report(key) for key in ["customer_id", "country", "status"]}
for key, (used, max_rows, skew) in report.items():
    print(f"DISTRIBUTED BY ({key:<11}) segments used={used} max rows/segment={max_rows:>7,} skew={skew}")

assert report["customer_id"][0] == SEGMENTS and report["customer_id"][2] < 0.1  # high cardinality, even
assert report["country"][0] <= 4 and report["country"][2] > 0.5                 # 4 values: most segments idle
assert report["status"][0] <= 2                                                  # 2 values: at most 2 busy segments

### Motion and distributed joins

The optimizer works out where rows must flow before an operation can run. If two tables are hash-distributed on the join key, the join is segment-local. Otherwise the planner inserts a motion: a Redistribute motion hashes rows on the join key, or a Broadcast motion copies the small input to every segment. An aggregate that cannot be partially computed locally gets a Gather motion back to the coordinator.

Motion means rows cross the interconnect, so it is the dominant cost in most bad Greenplum plans. Look at `EXPLAIN` output for `Motion` nodes and compare `rows` between the sender and receiver. Choosing the distribution key that matches the busiest join removes an entire shuffle.

```sql
EXPLAIN
SELECT c.country, sum(o.amount)
FROM fact_orders o
JOIN dim_customer c ON c.customer_id = o.customer_id
GROUP BY c.country;
```

Yes, and the planner picks one of three strategies per join. If both inputs are hash-distributed on the join key, the join runs locally with no motion. If one side is small, it is broadcast to every segment so each segment joins its local rows against the full copy. Otherwise both sides are redistributed on the join key, or the smaller side is redistributed and the other stays put.

```sql
EXPLAIN
SELECT s.order_id, c.name
FROM fact_sales s
JOIN dim_customer c ON c.customer_id = s.customer_id;
```

Read the plan for `Redistribute Motion` or `Broadcast Motion` above the join. Broadcast is cheap only when the broadcast side is genuinely small; the planner decides from `reltuples`, so stale statistics can make it broadcast a large table and flood the interconnect. Replicated tables eliminate the motion entirely because a full copy already exists on each segment.

Common mistake: joining on a non-distribution key and assuming the planner will sort it out. It will, but with a full shuffle every time; matching the keys removes the shuffle.

In [ ]:
def rows_moved(fact_key, strategy):
    """Rows crossing the interconnect to join fact_orders with dim_customer on customer_id."""
    if strategy == "colocated":  # both sides already hashed on the join key
        return con.execute(f"""
            SELECT count(*) FROM fact_orders
            WHERE {segment_of(fact_key)} <> {segment_of('customer_id')}
        """).fetchone()[0]
    if strategy == "redistribute":  # Redistribute Motion: rehash fact rows on customer_id
        return con.execute(f"""
            SELECT count(*) FROM fact_orders
            WHERE {segment_of(fact_key)} <> {segment_of('customer_id')}
        """).fetchone()[0]
    if strategy == "broadcast":  # Broadcast Motion: copy the whole dimension to every other segment
        return con.execute("SELECT count(*) FROM dim_customer").fetchone()[0] * (SEGMENTS - 1)
    raise ValueError(strategy)


colocated = rows_moved("customer_id", "colocated")          # fact DISTRIBUTED BY (customer_id)
redistributed = rows_moved("order_id", "redistribute")      # fact DISTRIBUTED BY (order_id)
broadcast = rows_moved("order_id", "broadcast")

assert colocated == 0
assert broadcast < redistributed  # a 5,000-row dimension is cheaper to copy than 200,000 fact rows to move
print(f"rows moved: colocated={colocated:,} broadcast dim={broadcast:,} redistribute fact={redistributed:,}")

revenue_by_tier = con.execute("""
    SELECT d.tier, sum(f.amount) FROM fact_orders f JOIN dim_customer d USING (customer_id)
    GROUP BY 1 ORDER BY 1
""").fetchall()
assert len(revenue_by_tier) == 4  # the answer is identical whatever the motion; only the cost differs

### Detecting and fixing skew

Data skew is an uneven number of rows per segment. It comes from a distribution key with a few dominant values, or from a small number of very large groups in a `GROUP BY`. Hotspots are the operator-level version of the same problem: one slice finishes long after the others because one segment holds most of the rows it processes.

The fixes are to change the distribution key to a higher-cardinality column, or to use `DISTRIBUTED RANDOMLY` when no key is dominant, or to salt the key with a computed bucket. For group skew, pre-aggregate or add a bucket column and aggregate in two phases.

```sql
SELECT * FROM gp_toolkit.gp_skew_coefficients
ORDER BY abs(skccoeff) DESC LIMIT 10;

SELECT * FROM gp_toolkit.gp_skew_idle_fractions
ORDER BY abs(siffraction) DESC LIMIT 10;
```

Common mistake: fixing skew by adding a segment. More segments do not help when one key value still hashes to one place; they add idle capacity.

### Star schemas

Star schemas are the ideal Greenplum shape. Distribute the fact on the busiest foreign key, replicate or distribute the dimensions on their primary keys, and the dominant joins become segment-local. Snowflake schemas add second-level dimensions, which introduce more joins; keep those small and replicated, or denormalize them into the star when query patterns allow.

```sql
CREATE TABLE fact_sales (
  sale_id bigint, sale_date date, product_id int, store_id int, revenue numeric(12,2)
) DISTRIBUTED BY (product_id)
WITH (appendoptimized=true, orientation=column, compresstype=zstd);

CREATE TABLE dim_product (product_id int, category text) DISTRIBUTED BY (product_id);
CREATE TABLE dim_store   (store_id int, region text)     DISTRIBUTED REPLICATED;
```

With the fact and `dim_product` colocated, that join has no motion. `dim_store` is replicated, so the store join has no motion either. Partition the fact by `sale_date` for pruning.

Common mistake: distributing the fact on a surrogate primary key such as `sale_id`. Every dimension join then redistributes, and the schema loses its main MPP advantage.

## Storage

### Heap, append-optimized and column orientation

Storage is a per-table decision with three independent axes: heap versus append-optimized, row versus column orientation, and compression per column. The fourth and fifth decisions are partitioning and indexes.

Use heap row storage for tables with frequent single-row updates or deletes. Use append-optimized row storage for append-only data written and read as whole rows, such as log lines. Use append-optimized column storage for wide analytical tables where queries touch a few columns. Add compression per column: `zstd` as the general default, `rle` for sorted or low-cardinality columns, `zlib` where ratio matters more than speed, `none` for a column that compresses badly.

```sql
CREATE TABLE fact_events (
  event_id   bigint,
  event_date date,
  user_id    bigint,
  event_type text,
  payload    text
)
WITH (appendoptimized=true, orientation=column, compresstype=zstd, compresslevel=3)
DISTRIBUTED BY (user_id)
PARTITION BY RANGE (event_date)
  (START (date '2026-01-01') END (date '2027-01-01') EVERY (interval '1 month'),
   DEFAULT PARTITION p_default);

ALTER TABLE fact_events ALTER COLUMN event_type SET COMPRESSTYPE rle;
```

Common mistake: putting a heavily updated table in column store. An `UPDATE` on AOCO rewrites blocks and marks the old rows dead, so bloat grows faster than vacuum can clear it

### Compression

AO tables compress per column and per block, and Greenplum picks the algorithm per column when you specify `compresstype`. `zstd` gives the best ratio-to-speed tradeoff and is the modern default recommendation. `zlib` compresses more slowly with a similar or slightly better ratio and is the legacy default. `quicklz` is fast with a modest ratio. `rle` is run-length encoding and is excellent for sorted or low-cardinality columns, but poor for high-cardinality ids. `none` disables compression for a column.

```sql
CREATE TABLE fact_logs (
  log_id   bigint,
  log_date date,
  level    text,
  message  text
)
WITH (appendoptimized=true, orientation=row, compresstype=zstd, compresslevel=1)
DISTRIBUTED BY (log_id);

ALTER TABLE fact_logs ALTER COLUMN level SET COMPRESSTYPE rle;
```

Compression is chosen at create time or changed with `ALTER TABLE ... SET COMPRESSTYPE`. `ANALYZE` does not change compression, and changing it rewrites the table. Note that compression is a storage decision: it trades CPU on every read for fewer bytes on disk and less interconnect traffic.

Common mistake: choosing the highest compression level everywhere. Decompression CPU then dominates scan time, and the reading query gets slower, not faster.

Sort order matters as much as the codec: low-cardinality columns sorted into long runs compress far better.

In [ ]:
workdir = Path(tempfile.mkdtemp(prefix="greenplum_storage_"))


def file_size(label, order_by, codec):
    path = workdir / f"events_{label}_{codec}.parquet"
    con.execute(f"""
        COPY (SELECT country, status FROM fact_orders ORDER BY {order_by})
        TO '{path}' (FORMAT parquet, COMPRESSION {codec})
    """)
    return path.stat().st_size


sizes = {
    "unsorted, uncompressed": file_size("unsorted", "hash(order_id)", "uncompressed"),
    "unsorted, zstd": file_size("unsorted", "hash(order_id)", "zstd"),
    "sorted by country/status, zstd": file_size("sorted", "country, status", "zstd"),
}
for label, size in sizes.items():
    print(f"{label:<32} {size / 1024:8.1f} KiB")

assert sizes["unsorted, zstd"] < sizes["unsorted, uncompressed"]
assert sizes["sorted by country/status, zstd"] < sizes["unsorted, zstd"]  # long runs: RLE and dictionaries shine

### Partitioning and partition elimination

Partitioning splits one logical table into child tables selected by a partition key. Greenplum supports range partitioning, list partitioning, and multi-level combinations. Range partitions can be declared with `START`, `END`, and `EVERY`, or named individually. A `DEFAULT PARTITION` catches rows that match no declared partition.

```sql
CREATE TABLE fact_sales (
  sale_id    bigint,
  sale_date  date,
  region     text,
  revenue    numeric(12,2)
)
DISTRIBUTED BY (sale_id)
PARTITION BY RANGE (sale_date)
  (START (date '2026-01-01') END (date '2027-01-01') EVERY (interval '1 month'),
   DEFAULT PARTITION p_default);
```

At plan time the optimizer performs partition pruning: a filter on `sale_date` eliminates child tables that cannot contain matching rows, so only those children are scanned. The default partition is always scanned unless it can be excluded, so keep it small or split it periodically.

Common mistake: partitioning on a column that is never filtered. If no query constrains the partition key, pruning never happens and partitioning only adds planning overhead.

Partition elimination, also called pruning, happens at plan time. When a query constrains the partition key, the planner examines the partition definitions and excludes child tables that cannot contain matching rows. Only the surviving children are scanned, so a monthly query on a five-year table reads a fraction of the data.

```sql
EXPLAIN
SELECT sum(revenue)
FROM fact_sales
WHERE sale_date >= date '2026-06-01'
  AND sale_date <  date '2026-07-01';
```

The plan lists only `fact_sales_1_prt_...` children whose range overlaps the filter. Pruning works on constants and on stable expressions, but a function on the partition column, such as `date_trunc('month', sale_date)`, can prevent it. Write the predicate so the raw column compares directly to a constant or parameter.

Common mistake: keeping a large default partition and filtering it. Rows that fail to match any declared partition land there, and the default is scanned whenever it cannot be excluded.

Partitioning pays when queries filter on the partition key, because pruning removes whole child tables at plan time. It costs planning time and management overhead, so it is not free. A rule of thumb is that pruning must remove at least half the data for the common query to be worth the machinery.

Bitmap indexes suit append-optimized tables with low-cardinality columns, because they are compact and combine multiple predicates efficiently. B-tree indexes suit high-cardinality equality and range lookups on heap tables. BRIN indexes are cheap and effective on naturally ordered columns such as timestamps.

```sql
CREATE INDEX idx_events_type ON fact_events USING bitmap (event_type);
CREATE INDEX idx_events_ts   ON fact_events USING brin (event_date);

EXPLAIN
SELECT event_type, count(*)
FROM fact_events
WHERE event_date >= date '2026-06-01' AND event_date < date '2026-07-01'
GROUP BY event_type;
```

The plan should list only the June child plus any surviving default partition. If the filter applies a function to `event_date`, pruning usually does not happen; compare the raw column against constants instead

### VACUUM and compaction

Heap tables accumulate dead tuples and need `VACUUM` to mark space reusable; `VACUUM FULL` rewrites the table to return space to the operating system but takes an exclusive lock. AO and AOCO tables track deletes in a visibility map rather than on the data blocks, so a deleted row still occupies its block until the table is rewritten. `VACUUM` on an AO table updates the visibility map so the block can be skipped, and a table rewrite or `VACUUM FULL` physically reclaims it.

```sql
VACUUM fact_orders;                -- heap and AO: make space reusable, update visibility
VACUUM ANALYZE fact_orders;        -- vacuum then refresh statistics
VACUUM FULL fact_orders;           -- rewrite and reclaim, exclusive lock
ANALYZE fact_orders;               -- statistics only, cheap and safe
```

You can see bloat with `gp_toolkit.gp_bloat_diag`. After a large delete or update on an AO table, a rewrite is often the only way to shrink the file.

Common mistake: running `VACUUM FULL` on a busy table during the day. It takes an `ACCESS EXCLUSIVE` lock and blocks all readers and writers until the rewrite finishes.

## Loading and external data

### Parallel loads with gpfdist

The fastest path is an external table over `gpfdist`. `gpfdist` is a file server that runs on the segment hosts, reads local files, and streams them in parallel to every segment. Loading then becomes a `SELECT ... FROM external_table` or an `INSERT INTO target SELECT * FROM external_table`.

```bash
gpfdist -d /data/load/orders -p 8080 -l /var/log/gpfdist.log &
```

```sql
CREATE EXTERNAL TABLE ext_orders (
  order_id bigint, customer_id bigint, order_ts timestamp, amount numeric(12,2)
)
LOCATION ('gpfdist://etl-host:8080/orders/*.csv')
FORMAT 'csv' (header)
LOG ERRORS SEGMENT REJECT LIMIT 10;

INSERT INTO fact_orders SELECT * FROM ext_orders;
```

The key sizing rule is one file per segment or a few, so every segment reads its own file in parallel. Compression on the source files reduces network traffic. `gpload` wraps this in YAML, and `COPY` works but funnels data through the coordinator.

Common mistake: loading a single 500 GB file. One segment reads it while the rest idle, and the coordinator splits the stream.

Loads are tuned for parallelism. Use `gpfdist` over an external table so every segment reads its own files, size the files so each segment gets a few substantial ones, and compress the source files. Avoid `COPY` for large loads because it funnels data through the coordinator and the segments sit idle.

For backup, `gpbackup` runs in parallel across segments; increase `--jobs` to match the cluster and write to storage that every segment host can reach. Use `--leaf-partition-data` so a partition can be restored alone, and `--single-data-file` to reduce small-file overhead.

```bash
gpfdist -d /data/load/orders -p 8080 -l /var/log/gpfdist.log &

gpbackup --dbname analytics --backup-dir /backups --jobs 16 --leaf-partition-data
gprestore --timestamp 20261007 --backup-dir /backups --jobs 16 --create-db
```

```sql
CREATE EXTERNAL TABLE ext_orders (
  order_id bigint, customer_id bigint, order_ts timestamp, amount numeric(12,2)
)
LOCATION ('gpfdist://etl-host:8080/orders/*.csv')
FORMAT 'csv' (header)
LOG ERRORS SEGMENT REJECT LIMIT 100;
```

Common mistake: one huge input file. Only the segment that reads it works, and the coordinator may have to split the stream, so the cluster is effectively single-threaded for the load

### Handling bad rows during loads

External tables accept `LOG ERRORS SEGMENT REJECT LIMIT`, which captures rows that fail to parse or cast into an error table instead of aborting the load. It can be a count or a percentage. The error table is named `ext_table_err` and records the original row plus a message, so a load can tolerate dirty data and you can inspect what was rejected.

```sql
CREATE EXTERNAL TABLE ext_orders (
  order_id bigint, customer_id bigint, order_ts timestamp, amount numeric(12,2)
)
LOCATION ('gpfdist://etl-host:8080/orders/*.csv')
FORMAT 'csv' (header)
LOG ERRORS SEGMENT REJECT LIMIT 100;

INSERT INTO fact_orders SELECT * FROM ext_orders;

SELECT * FROM ext_orders_err LIMIT 10;
```

For strict pipelines use `LIMIT 0` so any rejected row fails the load. Add constraints and a validation `SELECT` with `WHERE` predicates before insert for business rules that parsing cannot catch.

Common mistake: setting a high reject limit and never checking the error table. Bad rows disappear silently and downstream totals stop reconciling.

### PXF, Parquet and object storage

Yes, through external tables and PXF. An external table is a thin definition over files that live outside the cluster; reads and writes go through `gpfdist`, `file`, or a PXF protocol for S3, HDFS, and object stores. PXF pushes projection and predicate work where the format allows, and reads Parquet, ORC, Avro, JSON, and text.

```sql
CREATE EXTERNAL TABLE ext_s3_orders (
  order_id bigint, order_ts timestamp, amount numeric(12,2)
)
LOCATION ('pxf://acme-lake/orders/*.parquet?PROFILE=s3:parquet&SERVER=default')
FORMAT 'CUSTOM' (FORMATTER='pxfwritable_import');
```

External storage is best for cold data, staging, and one-off joins with local tables. It is not a substitute for cluster storage when a table is scanned constantly, because every read pays network and object-store latency.

Common mistake: joining two external tables repeatedly. Each scan re-reads the remote files, so materialize the repeated one as an AO column table instead.

## Query planning

### Reading EXPLAIN and EXPLAIN ANALYZE

`EXPLAIN` prints the plan without running it. Read it from the bottom up. Scans are at the leaves, the final `Gather Motion` is usually at the top, and each node carries an estimated cost and row count. Greenplum adds `Slice` numbers: nodes sharing a slice run together on every segment, and a motion node marks where rows move between slices.

The four things to look for are the scan type on the largest table, the join method and which side is the build input, every motion node and its direction, and whether row estimates look plausible against the table's real size. Motion is the keyword unique to MPP; `Redistribute Motion` hashes rows on a key, `Broadcast Motion` copies rows to every segment, and `Gather Motion` collects results on the coordinator.

```sql
EXPLAIN
SELECT c.country, sum(o.amount) AS revenue
FROM fact_orders o
JOIN dim_customer c ON c.customer_id = o.customer_id
WHERE o.order_ts >= date '2026-01-01'
GROUP BY c.country;
```

A broadcast of a table that is not small is the classic red flag: the sender ships the whole relation to every segment, and the interconnect becomes the bottleneck

`EXPLAIN ANALYZE` runs the query and adds actual rows, actual time, and per-slice statistics. Greenplum reports the maximum across segments for a slice, so a large gap between the slowest and the average is skew. A large gap between estimated and actual rows means stale or missing statistics, and the plan may already be wrong because of it.

```text
Gather Motion 3:1  (slice1; segments: 3)  (cost=... rows=1 width=24)
  ->  HashAggregate  (cost=... rows=1 width=24)
        Group Key: c.country
        ->  Hash Join  (cost=... rows=1000000 width=20)
              Hash Cond: (o.customer_id = c.customer_id)
              ->  Seq Scan on fact_orders o  (slice0)
                    Filter: (order_ts >= '2026-01-01'::date)
              ->  Hash  (slice0)
                    ->  Seq Scan on dim_customer c
```

Slice numbers tell you how many parallel passes the plan has. When one slice's `rows` maximum is far above two times its average, redistribute or re-partition the data feeding that operator. When `actual rows` is orders of magnitude off from the estimate, run `ANALYZE` on every table in the join before changing anything else.

Common mistake: reading only the total runtime. The reason a query is slow lives in the slowest slice, and the total hides it

## Workload management and concurrency

### Resource groups

Resource groups are the modern mechanism. A resource group is a named bucket of CPU, memory, and concurrency that you assign to a role or to the whole database. When a role with a group runs a query, the query is admitted only if the group has a free concurrency slot and enough memory, and its CPU share is enforced by the Linux cgroup on each segment host.

```sql
CREATE RESOURCE GROUP rg_etl WITH (
  CONCURRENCY=4,
  CPU_RATE_LIMIT=40,
  MEMORY_LIMIT=40,
  MIN_COST=0
);
CREATE ROLE etl_user LOGIN RESOURCE GROUP rg_etl;
```

`MEMORY_LIMIT` caps memory per segment for the group. `CONCURRENCY` caps simultaneous queries per segment. Set one of `CPU_RATE_LIMIT` or `CPUSET` to pin CPUs. The admin group `gp_adhoc` and default group `gp_default_group` come preconfigured. `gp_toolkit.gp_resgroup_status` shows current usage, and `pg_resgroup` lists definitions.

Common mistake: giving ETL a large `MEMORY_LIMIT` and no concurrency cap. A few wide sort or hash operators then spill or trigger the segment memory limit and abort queries.

Segment memory is the scarce resource. Every slice of every concurrent query can hold sort and hash state, and when the total exceeds the segment's limit a query spills to disk or is terminated. Resource groups bound that risk by capping concurrency and memory per group on each segment.

```sql
CREATE RESOURCE GROUP rg_bi WITH (
  CONCURRENCY=20,
  CPU_RATE_LIMIT=30,
  MEMORY_LIMIT=25
);
CREATE RESOURCE GROUP rg_etl WITH (
  CONCURRENCY=3,
  CPU_RATE_LIMIT=45,
  MEMORY_LIMIT=55
);
ALTER ROLE bi_user  RESOURCE GROUP rg_bi;
ALTER ROLE etl_user RESOURCE GROUP rg_etl;

SELECT rsgname, num_running, num_queued, memory_used, cpu_usage
FROM gp_toolkit.gp_resgroup_status;
```

The memory limits across all groups must sum to 100 or less per segment, and the CPU shares likewise. A group that queues is healthier than a cluster that spills everywhere; admission control converts an unbounded slowdown into a predictable wait.

Common mistake: setting `MEMORY_LIMIT` from a single query's peak need and ignoring concurrency. Ten concurrent queries in the group each get a share of the group cap, so per-query memory is the cap divided by active concurrency

### Transactions and MVCC

Every statement that touches more than one segment runs as a distributed transaction coordinated by the coordinator. The coordinator assigns a global transaction id, starts a local transaction on each participating segment, and commits with two-phase commit: prepare on every segment, then commit if all prepared. If a segment fails during commit, the transaction is recovered from the coordinator's transaction log.

Reads use MVCC snapshots consistent with PostgreSQL, so a query sees one snapshot across all segments. The coordinator's `gxid` and the distributed log are what make that possible.

```sql
BEGIN;
UPDATE accounts SET balance = balance - 100 WHERE id = 1;
UPDATE accounts SET balance = balance + 100 WHERE id = 2;
COMMIT;
```

Two-phase commit means a distributed commit is more expensive than a local one. Keeping transactions short reduces the window in which a segment failure triggers recovery.

Common mistake: holding a distributed transaction open while waiting on an external service. That blocks vacuum and inflates `pg_stat_activity` age across the cluster.

### Statistics and maintenance

Statistics are what the planner believes. `ANALYZE` refreshes row counts, distinct counts, and histograms; run it after every large load, after a partition is added or dropped, and on any table whose data changed by more than a few percent. `EXPLAIN ANALYZE` is the check: when actual and estimated rows diverge by an order of magnitude, the plan was built on a bad estimate.

`VACUUM` is the maintenance counterpart. On heap tables it marks dead tuples reusable and updates the visibility map. On append-optimized tables it updates the visibility map so deleted blocks can be skipped, but physical space returns only with a rewrite. Autovacuum runs, but on a large warehouse it is often too slow to keep up with bulk deletes.

```sql
ANALYZE fact_orders;
VACUUM fact_orders;
VACUUM ANALYZE fact_orders;

SELECT relname, n_dead_tup, last_analyze, last_autoanalyze, last_vacuum, last_autovacuum
FROM pg_stat_user_tables
ORDER BY n_dead_tup DESC LIMIT 10;
```

Common mistake: treating `ANALYZE` as optional after a load. It is the cheapest performance action available and the most commonly skipped

## Operations

### High availability

High availability is built on mirrors. Every primary segment has a mirror on a different host, configured as a pair in `gp_segment_configuration`. The primary streams write-ahead log to its mirror. If a primary fails, the mirror is promoted to primary and the coordinator marks the old primary as down; the cluster keeps serving.

`gpstate -s` reports which segments are up, which are mirrors, and whether the cluster is in change tracking. `gprecoverseg` resynchronizes a recovered mirror with the changes it missed, avoiding a full rebuild.

```bash
gpstate -s
gpstate -e
gprecoverseg -a
```

Fault tolerance covers segment loss, not coordinator loss in the basic single-coordinator deployment. A coordinator failure stops the cluster until it is restored, so a standby coordinator is the usual next step for a production system.

Common mistake: assuming mirrors also protect against data corruption. Mirrors replicate, so a bad `DELETE` or a bad load propagates to the mirror instantly.

### Backup and restore

`gpbackup` is the modern parallel backup utility and replaces `gpcrondump`. It runs on the coordinator, connects to every segment, and writes each segment's data and metadata in parallel to a backup directory on a shared filesystem or to S3. `gprestore` reads that set and loads it back, also in parallel.

```bash
gpbackup --dbname analytics --backup-dir /backups --leaf-partition-data   --single-data-file --jobs 8

gprestore --timestamp 20261007 --backup-dir /backups --jobs 8
```

`--leaf-partition-data` backs up each partition as its own file, so a restore can target one partition. `--single-data-file` writes one file per segment instead of many small ones. `gpbackup --incremental --from-timestamp TIMESTAMP` backs up only files that changed since that backup and requires a supported storage plugin. `--metadata-only` captures DDL without data, and `--with-stats` includes query planner statistics.

Common mistake: testing the backup but never the restore. Restore time is the number that matters for recovery objectives, and it depends on parallelism, file count, and network bandwidth.

### Scaling with gpexpand

Scale out by adding segment hosts with `gpexpand`. It redistributes existing tables across the new segment count, which is a heavy, long-running operation but gives near-linear capacity and scan growth. The alternative is scale up: more CPU, memory, and faster storage per host, which helps when the bottleneck is per-segment.

```bash
gpexpand -f new_hosts_file            # write the expansion input file
gpexpand -i gpexpand_inputfile -n 8    # initialize new segments
gpexpand -d 60:00:00                   # redistribute for up to 60 hours
gpexpand -a                            # run ANALYZE after expansion
gpexpand -c                            # clean up the expansion schema
```

`gpexpand` rewrites every distributed table using the new segment count, so plan a maintenance window when writes are quiet. Replicated tables grow with the segment count, which is another reason not to replicate large tables.

Common mistake: adding segments to fix skew. If one distribution key value dominates, the new segments still receive none of that key's rows, so the hot segment stays hot.

## Security

Security has four layers. Authentication is PostgreSQL: `pg_hba.conf` controls which host, user, and database may connect and with which method, including LDAP, Kerberos, and SCRAM. Authorization is role-based access control with `GRANT` and `REVOKE` on schemas, tables, and columns. Row-level security policies restrict which rows a role can see. Encryption covers data at rest through filesystem or disk encryption, and in transit through SSL/TLS on connections.

```sql
CREATE ROLE analyst LOGIN;  -- authenticate via LDAP/Kerberos in pg_hba.conf, not a literal password
GRANT USAGE ON SCHEMA analytics TO analyst;
GRANT SELECT (order_id, order_ts, amount) ON fact_orders TO analyst;

CREATE POLICY tenant_policy ON fact_orders
  USING (tenant_id = current_setting('app.tenant_id')::int);
ALTER TABLE fact_orders ENABLE ROW LEVEL SECURITY;
```

SSL is enabled with `ssl=on` and certificates; `gp_toolkit` and `pg_stat_ssl` show connection encryption.

Common mistake: granting `SELECT` on the whole table when a subset of columns is enough. The grant is the easiest control to enforce and the easiest to over-apply.

Roles are cluster-wide in PostgreSQL, so a role created in one database can be granted access in another. A role can `LOGIN` or be a group for privileges. Greenplum adds a resource group or resource queue assignment to the role, plus `CREATEROLE`, `CREATEDB`, and superuser attributes.

```sql
CREATE ROLE analyst LOGIN RESOURCE GROUP rg_bi;  -- credentials come from LDAP/Kerberos
CREATE ROLE readers;
GRANT readers TO analyst;

GRANT USAGE ON SCHEMA analytics TO readers;
GRANT SELECT ON ALL TABLES IN SCHEMA analytics TO readers;
ALTER DEFAULT PRIVILEGES IN SCHEMA analytics GRANT SELECT ON TABLES TO readers;
```

`ALTER DEFAULT PRIVILEGES` is what makes new tables automatically readable by the group, avoiding a grant step after every deployment. Inspect effective access with `information_schema.role_table_grants` or `has_table_privilege`.

Common mistake: granting privileges to a login role directly. When team membership changes you must re-grant; grant to a group role and manage membership instead.

## Trade-offs: when not to use Greenplum

Greenplum is strong for large structured SQL workloads: scans, joins, aggregations, and in-database analytics on tens of terabytes and beyond, with PostgreSQL compatibility as a large practical bonus. The trade-offs are operational weight and a design burden. You must run and tune a cluster, choose distribution keys and storage for every table, and keep statistics fresh. Every statement is a distributed plan dispatched through the coordinator, so point lookups and single-row writes are slow.

It is a poor fit for OLTP, for small datasets that fit comfortably on one node, and for workloads that are dominated by very high-frequency small updates. It is also not the natural home for raw files that belong in a data lake, even though PXF can read them.

```text
Choose Greenplum for:      large scans, star-schema analytics, batch transformations, in-database ML.
Avoid Greenplum for:       OLTP, tiny datasets, streaming with sub-second reads, blob stores.
```

Common mistake: adopting an MPP warehouse for a workload a single PostgreSQL instance would serve better, then paying cluster operations for no gain.

## Things to remember

### Key numbers and limits

| Topic | Number / fact | Why it matters |
|---|---|---|
| Distribution | `DISTRIBUTED BY (key)` spreads rows across segments by hash; `DISTRIBUTED REPLICATED` copies small tables to every segment | distribution decides whether a join is local or redistributed over the network |
| Skew | a few percent skew is noise; a hot distribution key can make one segment do most of the work | check `gp_toolkit.gp_skew_coefficients`; one hot segment gates the whole query |
| Storage | heap (row, updatable) vs append-optimized (AO) row or column-oriented tables | AO/columnar with compression is the analytics default; heap suits heavy OLTP-style churn |
| Partitioning | range/list partitions with pruning; partition count multiplies metadata per table | partition on the filter column (usually date), not cardinality |
| Optimizer | GPORCA is the default cost-based optimizer; it depends on fresh statistics | run `ANALYZE` after loads — stale stats mean wrong join orders and redistribution |
| Vacuum | dead tuples from updates/deletes bloat heap tables | schedule `VACUUM` after churn; bloat makes every scan slower |
| Concurrency | resource groups (or legacy resource queues) cap memory and concurrency per role | BI storms need group isolation, not just more hardware |
| Interconnect | the private network fabric that redistributes and broadcasts rows during queries | large redistributions in the plan are the cost to hunt |

### Semantics to memorize

* **Distribution follows the dominant join.** Hash-distribute the large tables on the join key; replicate small dimensions so they never move.
* **Redistribution is the expensive motion.** Read EXPLAIN for Redistribute and Broadcast motions — Broadcast of a large table or Redistribute of a big fact is the smell.
* **Statistics drive everything.** GPORCA chooses join order, distribution strategy, and scan method from statistics; load tables, then ANALYZE, always.
* **Partition pruning happens at scan time.** A query without a predicate on the partition column scans every partition.
* **Loading in parallel is the design point.** External tables / gpfdist load straight into segments in parallel; single-connection INSERT is the slow path.
* **HA is segment mirroring.** A primary segment has a mirror on another host; failure over to doubles the load on the surviving host until rebalanced.

### Mistakes that keep coming back

* Distributing on the filter column instead of the join key — every join then redistributes one side.
* Random distribution for permanent fact tables — fine for staging, wrong for anything joined regularly.
* Never running ANALYZE after bulk loads — GPORCA plans against stale row counts.
* Letting heap tables bloat after big deletes — VACUUM is a maintenance habit, not a remediation.
* Partitioning a small table — thousands of partitions on a small table slows the planner more than pruning saves.
* Reading EXPLAIN row counts only — use `EXPLAIN ANALYZE` and compare rows per segment to find the hot one (skew).
* Loading through a single psycopg2 connection — use external tables or gpload so segments ingest in parallel.
* Ignoring spill files — memory-per-statement under-sized resource groups spill joins to disk.

## Real world use cases

### 1. Enterprise data warehouse consolidation

A retailer replaces several departmental warehouses with one Greenplum cluster: ~120 TB, ~40 billion fact rows, and daily loads from 30 source systems. Data is loaded with gpfdist, stored as AO/CO Parquet-like columnar tables, and partitioned by month.

```bash
gpfdist -d /stage/retail -p 8081 &
```

```sql
create table fact_sales (
  sale_id bigint, store_id int, product_id int, amount numeric, sale_date date
)
with (appendonly = true, orientation = column, compresstype = zstd)
distributed by (store_id)
partition by range (sale_date)
  (start (date '2026-01-01') every (interval '1 month') end (date '2027-01-01'));

select store_id, sum(amount) from fact_sales
where sale_date >= date '2026-10-01' group by store_id;
```

Why: parallel gpfdist loads, columnar compression, and partition pruning keep a month-scoped query on a huge fact table fast.

### 2. Multi-tenant SaaS analytics with resource isolation

A SaaS vendor serves 200 tenants from one cluster. Tenant data is separated by row-level security, and noisy tenants are isolated with resource groups. ~30 TB, ~5,000 concurrent dashboard queries during business hours.

```sql
alter table events enable row level security;
create policy tenant_isolation on events
  using (tenant_id = current_setting('app.tenant_id')::int);

create resource group rg_dashboard with (concurrency = 40, cpu_rate_limit = 40, memory_limit = 25);
create role dashboard_user resource group rg_dashboard;
```

Why: RLS keeps one physical table safe across tenants, and resource groups stop one tenant's heavy query from starving the rest. Row-level security (`CREATE POLICY`) is a Greenplum 7 feature; Greenplum 6 does not support it, so tenant isolation there relies on separate schemas or views.

### 3. Querying a data lake in place with PXF

A media company keeps 800 TB of Parquet on S3 and only loads aggregates into Greenplum. PXF external tables read S3 and HDFS without copying, while curated AO/CO tables hold the hot last 90 days.

```sql
create external table ext_watch_events (user_id bigint, content_id bigint, watched_ms bigint)
location ('pxf://media-lake/events/?PROFILE=s3:parquet')
format 'custom' (formatter = 'pxf');

insert into fact_watch_daily
select date_trunc('day', to_timestamp(watched_ms::double precision / 1000)) as day, content_id, count(*) as plays
from ext_watch_events group by 1, 2;
```

Why: PXF avoids duplicating a huge lake, and the warehouse keeps only the compressed, partitioned slice that dashboards query repeatedly.

## Greenplum exercises

These exercises are standard-library Python. They model the arithmetic behind Greenplum design choices: distribution keys, segment skew, motion cost, storage layout, compression, resource groups, and partition pruning. They are review exercises, not a connection to a live cluster. Run them with the repository virtual environment.

### Try it 1. Pick a distribution key from join patterns

**Task.** Write `pick_distribution_key(candidates)` where each candidate is `(name, join_share, distinct_keys, max_value_share)`. Choose the candidate with `join_share >= 0.5`, `distinct_keys >= 1000`, and `max_value_share <= 0.10`, preferring the highest join share. Return `"DISTRIBUTED RANDOMLY"` if none qualifies.

**Hint.** Filter first, then take the maximum by join share. Low cardinality and a dominant value both disqualify a key even when it appears in many joins.

**Expected result.** `customer_id`, and `DISTRIBUTED RANDOMLY` for the low-cardinality-only case.

In [ ]:
def pick_distribution_key(candidates):
    # Even spread plus the dominant join key. Discard low cardinality and skewed keys.
    usable = [
        (name, share)
        for name, share, distinct_keys, max_value_share in candidates
        if share >= 0.5 and distinct_keys >= 1000 and max_value_share <= 0.10
    ]
    if not usable:
        return "DISTRIBUTED RANDOMLY"
    return max(usable, key=lambda item: item[1])[0]


candidates = [
    ("order_id", 0.05, 500_000_000, 0.000000002),
    ("customer_id", 0.72, 4_000_000, 0.02),
    ("status", 0.60, 4, 0.55),
]

key = pick_distribution_key(candidates)
print("distribution key =", key)
print("fallback =", pick_distribution_key([("status", 0.90, 4, 0.60)]))

assert key == "customer_id"
assert pick_distribution_key([("status", 0.90, 4, 0.60)]) == "DISTRIBUTED RANDOMLY"
assert key != "order_id"

### Try it 2. Compute segment skew from row counts

**Task.** Given rows per segment, compute the coefficient of variation as the population standard deviation divided by the mean. Also compute the idle fraction as `1 - mean / max`. Flag the table when the coefficient is above `0.10`.

**Hint.** This mirrors `gp_toolkit.gp_skew_coefficients`, which compares the spread of per-segment row counts with the mean. A perfectly even table has a coefficient of zero.

**Expected result.** Coefficient 0.0112, idle fraction 0.0149, not flagged.

In [ ]:
import statistics

rows = [10_000_000, 9_800_000, 10_100_000, 9_900_000]

mean = statistics.mean(rows)
coefficient = statistics.pstdev(rows) / mean
idle_fraction = 1 - mean / max(rows)

print(f"skew coefficient = {coefficient:.4f}")
print(f"idle fraction = {idle_fraction:.4f}")
print("flagged =", coefficient > 0.10)

assert round(coefficient, 4) == 0.0112
assert round(idle_fraction, 4) == 0.0149
assert not coefficient > 0.10
assert min(rows) > 0

### Try it 3. Estimate motion cost from join columns

**Task.** Write `motion_seconds(rows, row_bytes, segments, network_gbps, kind)`. A redistribute sends each row once, so the copies factor is 1. A broadcast sends the rows to all other segments, so the factor is `segments - 1`. Compute total bytes and the transfer time.

**Hint.** `total_bytes = rows * row_bytes * copies`, then divide by `1e9` for gigabytes and by the network rate.

**Expected result.** Redistribute 4.8 seconds, broadcast 1.32 seconds. The small broadcast is much cheaper than the large redistribution, which is why the planner prefers it.

In [ ]:
def motion_seconds(rows, row_bytes, segments, network_gbps, kind):
    copies = segments - 1 if kind == "broadcast" else 1
    total_bytes = rows * row_bytes * copies
    return total_bytes / 1e9 / network_gbps


redistribute = motion_seconds(200_000_000, 48, 12, 2.0, "redistribute")
broadcast = motion_seconds(5_000_000, 48, 12, 2.0, "broadcast")

print(f"redistribute = {redistribute:.2f} s")
print(f"broadcast = {broadcast:.2f} s")

assert round(redistribute, 1) == 4.8
assert round(broadcast, 2) == 1.32
assert redistribute > broadcast

### Try it 4. Choose AO or column store from an access pattern

**Task.** Write `choose_storage(updates_per_day, columns_scanned, columns_total, append_only)`. Return `"HEAP"` when updates exceed 1000 per day or the table is not append-only. Return `"AO_COLUMN"` for append-only tables where at most a quarter of the columns are scanned. Return `"AO_ROW"` otherwise.

**Hint.** Updates decide heap versus append-optimized. The scanned-to-total column ratio decides column versus row orientation.

**Expected result.** `HEAP`, `AO_COLUMN`, `AO_ROW`, `HEAP` in that order.

In [ ]:
def choose_storage(updates_per_day, columns_scanned, columns_total, append_only):
    if not append_only or updates_per_day > 1000:
        return "HEAP"
    if columns_scanned / columns_total <= 0.25:
        return "AO_COLUMN"
    return "AO_ROW"


cases = [
    (choose_storage(5000, 3, 40, True), "HEAP"),
    (choose_storage(0, 3, 40, True), "AO_COLUMN"),
    (choose_storage(0, 35, 40, True), "AO_ROW"),
    (choose_storage(0, 3, 40, False), "HEAP"),
]

for got, expected in cases:
    print(got)

for got, expected in cases:
    assert got == expected

assert choose_storage(0, 10, 40, True) == "AO_COLUMN"

### Try it 5. Compute compression savings from a table profile

**Task.** A profile maps each column to `(uncompressed_gb, compression_ratio)`. Compute the stored size as the sum of `uncompressed_gb / ratio` and the savings fraction. Skip a column whose ratio is `1.0` because that means no compression.

**Hint.** Size, not ratio, is what you sum. Report the savings as `1 - stored / raw`.

**Expected result.** Stored 2.049 GB from 6.0 GB raw, savings 0.659.

In [ ]:
profile = {
    "event_id": (2.0, 1.8),
    "event_type": (0.5, 8.0),
    "payload": (3.5, 4.0),
}

raw_gb = sum(size for size, _ in profile.values())
stored_gb = sum(size / ratio for size, ratio in profile.values())
savings = 1 - stored_gb / raw_gb

print(f"raw = {raw_gb:.1f} GB")
print(f"stored = {stored_gb:.3f} GB")
print(f"savings = {savings:.3f}")

assert round(stored_gb, 3) == 2.049
assert round(savings, 3) == 0.659
assert stored_gb < raw_gb

### Try it 6. Size resource groups from concurrent workloads

**Task.** For each workload `(arrival_per_second, avg_seconds, memory_mb_per_query)`, compute the concurrency with Little's law and round up to a slot count. Convert each group's memory to a percentage of a `32768` MB segment, and check that the percentages sum to 100 or less.

**Hint.** `concurrency = arrival_rate * avg_seconds` and `slots = ceil(concurrency)`. Group memory is `slots * memory_mb_per_query`.

**Expected result.** BI 30 slots at 23.4 percent, ETL 3 slots at 37.5 percent, total 60.9 percent.

In [ ]:
import math

segment_mem_mb = 32768

workloads = {
    "bi": (2.0, 15.0, 256),
    "etl": (0.05, 60.0, 4096),
}

slots = {}
memory_pct = {}

for name, (arrival, avg_seconds, mem_mb) in workloads.items():
    concurrency = arrival * avg_seconds
    slots[name] = math.ceil(concurrency)
    memory_pct[name] = slots[name] * mem_mb / segment_mem_mb * 100

total_pct = sum(memory_pct.values())

for name in workloads:
    print(f"{name}: slots={slots[name]} memory={memory_pct[name]:.1f}%")
print(f"total memory = {total_pct:.1f}%")

assert slots["bi"] == 30
assert slots["etl"] == 3
assert round(memory_pct["bi"], 1) == 23.4
assert round(memory_pct["etl"], 1) == 37.5
assert round(total_pct, 1) == 60.9
assert total_pct <= 100

### Try it 7. Plan partition pruning from filters

**Task.** Build twelve monthly range partitions for 2026 and count how many overlap a filter from 2026-06-15 to 2026-09-10. Report the scanned and pruned counts and the scanned fraction.

**Hint.** A partition `[start, end)` overlaps an inclusive filter `[lo, hi]` when `start <= hi` and `end > lo`.

**Expected result.** 4 partitions scanned, 8 pruned, scanned fraction 0.333.

In [ ]:
from datetime import date

partitions = []
for month in range(1, 13):
    start = date(2026, month, 1)
    end = date(2027, 1, 1) if month == 12 else date(2026, month + 1, 1)
    partitions.append((start, end))

lo = date(2026, 6, 15)
hi = date(2026, 9, 10)

scanned = [p for p in partitions if p[0] <= hi and p[1] > lo]
pruned = len(partitions) - len(scanned)

print("scanned partitions =", len(scanned))
print("pruned partitions =", pruned)
print(f"scanned fraction = {len(scanned) / len(partitions):.3f}")

assert len(scanned) == 4
assert pruned == 8
assert round(len(scanned) / len(partitions), 3) == 0.333
assert len(partitions) == 12